In [ ]:
from _utils_siamese import *

In [ ]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

In [ ]:
def load_train_split(train_file):
    train_split = pd.read_csv(train_file)
    return train_split

def load_test_split(test_file):
    test_split = pd.read_csv(test_file)
    return test_split

Class_num = 2
Size = (64, 64, 64) 
Batchsize = 8
dropout_rate = 0.5
weight_decay = 1e-4
adam = Adam(learning_rate=0.0001, beta_1=0.9, beta_2=0.999, epsilon=1e-08, decay=0.0)

# Parameters
params = {'dim': Size,
          'batch_size': Batchsize,
          'n_classes': Class_num,
          'n_channels': 1,
          'shuffle': True}

params_val = {'dim': Size,
              'batch_size': 1,
              'n_classes': Class_num,
              'n_channels': 1,
              'shuffle': False}

In [ ]:
# Function to extract patient ID from file path
def extract_patient_id(file_path):

    file_name = file_path.split('/')[-1]
    patient_id = file_name.split('_')[0]
    return patient_id

# Load model and extract features, then store them into a CSV file
def load_model_and_extract_features(fold_num, generator, data_df, data_type='train'):

    fold_dir = f'.../fold_{fold_num}/'

    # Load the best model
    model_path = fold_dir + f'final_epoch_fold{fold_num}_model.h5'
    model = build_siamese_CNN(Size, weight_decay, dropout_rate, Class_num)
    model.load_weights(model_path)

    # Check if the specified layer exists
    try:
        intermediate_layer_model = Model(inputs=model.input, outputs=model.layers[-3].output)
        # intermediate_layer_model = Model(inputs=model.input, outputs=model.input)
    except ValueError as e:
        print(f"Error: The specified layer '{layer_name}' does not exist in the model. Available layers:")
        for layer in model.layers:
            print(layer.name)
        raise e
    
    # Extract features for each batch of data
    features = []
    for i in range(len(generator)):
        input_data, labels = generator[i][0], generator[i][1]  # Get input data (X_pre, X_post) and labels
        batch_features = intermediate_layer_model.predict(input_data)

        batch_features = np.array(batch_features) 
        # batch_features_flattened = batch_features.reshape(batch_features.shape[0], -1)   
        # features.append(batch_features_flattened)   ## Extract input features, add this step
        features.append(batch_features)
    
    # Stack all batch features together
    features = np.vstack(features)

    # Build DataFrame for the features
    patient_ids = data_df['Pre_path'].apply(extract_patient_id).values  
    binary_labels = data_df['Binary_Label'].values 
    feature_columns = [f'feature_{i}' for i in range(features.shape[1])]  # Generate column names based on feature dimensions
    df_features = pd.DataFrame(features, columns=feature_columns)

    df_final = pd.DataFrame({
        'Patient_ID': patient_ids,
        'Binary_Label': binary_labels
    }).join(df_features)

    csv_file_path = f'.../deep_features_fold{fold_num}_{data_type}.csv'
    df_final.to_csv(csv_file_path, index=False)
    print(f"{data_type.capitalize()} deep features saved to {csv_file_path}")
    
    return df_final

# Set fold number
fold_num = 4

# Load training, validation, and test data
train_split = load_train_split(".../STRANet/Data/train_data.csv")
test_split = load_test_split(".../STRANet/Data/test_data.csv")

# Data generators
training_generator = DataGenerator(train_split[['Pre_path', 'Post_pseudo_path', 'Binary_Label']], **params_val)  
test_generator = DataGenerator(test_split[['Pre_path', 'Post_pseudo_path', 'Binary_Label']], **params_val)

# Extract deep learning features from the training and testing sets and save to CSV
train_features_df = load_model_and_extract_features(fold_num, training_generator, train_split, data_type='train')
test_features_df = load_model_and_extract_features(fold_num, test_generator, test_split, data_type='test')